# 02 — Model Benchmarking

**Purpose:** Compare trained models against climatology and persistence baselines across all three lead times.

Run `enso train-model` for all targets before opening this notebook.

Contents:
1. Load metrics from JSON reports
2. Comparison table (accuracy + F1 macro)
3. Lead-time performance curves
4. Confusion matrices for best model
5. SHAP feature importance (LightGBM)


In [ ]:
import sys
sys.path.insert(0, '../..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.evaluation.metrics import results_to_dataframe
from src.evaluation.plots import plot_lead_time_comparison, plot_confusion_matrix

METRICS_DIR = Path('../../outputs/metrics')
FIGURES_DIR = Path('../../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load all metric JSON files
all_results = {}
for f in sorted(METRICS_DIR.glob('*.json')):
    with open(f) as fh:
        data = json.load(fh)
    all_results.update(data)

df_metrics = results_to_dataframe(all_results)
df_metrics

## Lead-time performance curves

In [ ]:
fig = plot_lead_time_comparison(df_metrics, metric='f1_macro',
                                 save_path=FIGURES_DIR / 'lead_time_f1_macro.png')
plt.show()

In [ ]:
fig = plot_lead_time_comparison(df_metrics, metric='accuracy',
                                 save_path=FIGURES_DIR / 'lead_time_accuracy.png')
plt.show()

## Confusion matrices — LightGBM

In [ ]:
for target in ['enso_t1', 'enso_t3', 'enso_t6']:
    if target in all_results and 'lightgbm' in all_results[target]:
        cm = all_results[target]['lightgbm']['confusion_matrix']
        fig = plot_confusion_matrix(cm, title=f'LightGBM — {target}',
                                    save_path=FIGURES_DIR / f'cm_{target}_lgbm.png')
        plt.show()

## SHAP feature importance

In [ ]:
import pandas as pd
shap_path = Path('../../outputs/metrics/shap_importance.csv')
if shap_path.exists():
    shap_df = pd.read_csv(shap_path)
    ax = shap_df.head(20).set_index('feature')['mean_abs_shap'].sort_values().plot(
        kind='barh', figsize=(8, 7), color='#FF5722', edgecolor='black', linewidth=0.4
    )
    ax.set_title('SHAP feature importance (LightGBM, mean |SHAP|)', fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'shap_global_importance.png')
    plt.show()
else:
    print('Run enso train-model first to generate SHAP values')